# Building a Multi-Agent Research Assistant with OpenAI Agents SDK and Olostep

This notebook builds a beginner-friendly multi-agent research assistant using:

- **OpenAI Agents SDK** for orchestration, specialist agents, tools, and structured outputs.
- **Olostep Python SDK** for Answer API, Search, Search with Scrape, and Scrape.

The final output is a **proper Markdown research report** for any user-provided research question, not just a short answer.


## Architecture

The OpenAI Agents SDK can orchestrate agents. In this notebook the **manager agent** is the orchestrator. It can call:

- `answer_query()` for the first fast answer from Olostep Answer API.
- A **judge agent** exposed as a tool with `judge_agent.as_tool(...)`.
- A **source research agent** exposed as a tool with `source_research_agent.as_tool(...)`.
- An **analyst agent** exposed as a tool with `analyst_agent.as_tool(...)`.

That means the notebook calls only one top-level agent run:

```python
result = await Runner.run(manager_agent, prompt)
```

The manager decides when to call the other agents and tools.


## Required Research Workflow

The manager agent is instructed to follow this workflow:

1. Try Olostep Answer API first with `answer_query()`.
2. Use the judge agent to decide if the answer is good enough.
3. If good, ask the analyst agent to turn it into a Markdown research report.
4. If not, use the source research agent. That agent first tries `search_with_scrape()`.
5. If still weak, the source research agent runs multiple targeted `search_web()` calls, selects important URLs, calls `scrape_url()`, and writes source notes.
6. The analyst agent synthesizes a final Markdown report.

Only these Olostep functions are used:

- `answer_query()`
- `search_web()`
- `search_with_scrape()`
- `scrape_url()`


## Setup

Run this once. The `-U` matters because older `olostep` versions did not include `client.searches`.


In [1]:
# %pip install -q -U openai-agents olostep python-dotenv pydantic


## Environment Variables

Create a `.env` file next to this notebook:

```bash
OPENAI_API_KEY=your_openai_api_key
OLOSTEP_API_KEY=your_olostep_api_key
RUN_LIVE_EXAMPLE=false
```

Set `RUN_LIVE_EXAMPLE=true` only when you want the example cells to call paid APIs. After the install cell upgrades packages, restart the notebook kernel if `olostep version` still prints an older version such as `1.0.4`.


In [2]:
import importlib.metadata
import json
import os
import textwrap
import warnings
from typing import Any

from dotenv import load_dotenv
from IPython.display import Markdown, display
from olostep import Olostep
from pydantic import BaseModel, Field

from agents import Agent, Runner, custom_span, flush_traces, function_tool, gen_trace_id, trace

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OLOSTEP_API_KEY = os.getenv("OLOSTEP_API_KEY")
RUN_LIVE_EXAMPLE = os.getenv("RUN_LIVE_EXAMPLE", "false").lower() == "true"

missing = [name for name, value in {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "OLOSTEP_API_KEY": OLOSTEP_API_KEY,
}.items() if not value]

if missing:
    print("Missing environment variables:", ", ".join(missing))
else:
    print("Environment loaded.")
print("RUN_LIVE_EXAMPLE =", RUN_LIVE_EXAMPLE)
print("olostep version =", importlib.metadata.version("olostep"))
print("agents tracing enabled. Traces appear in the OpenAI dashboard when OPENAI_API_KEY is set.")

# Olostep may add response fields before the SDK models are updated.
# Keep notebook output focused on agent traces and report content.
warnings.filterwarnings("ignore", message=".*extra field.*SDK model.*", category=UserWarning, module="olostep.*")


Environment loaded.
RUN_LIVE_EXAMPLE = True
olostep version = 1.1.0
agents tracing enabled. Traces appear in the OpenAI dashboard when OPENAI_API_KEY is set.


## Olostep SDK Search With Scrape Demo

This standalone cell uses the exact SDK style requested. It is guarded by `RUN_LIVE_EXAMPLE` so the notebook can run top-to-bottom without spending API credits.


In [3]:
if not OLOSTEP_API_KEY:
    print("Skipping SDK demo because OLOSTEP_API_KEY is missing.")
elif not RUN_LIVE_EXAMPLE:
    print("Skipping SDK demo. Set RUN_LIVE_EXAMPLE=true in .env to run it.")
    print("SDK call: client.searches.create(query=..., limit=5, scrape_options={'formats': ['markdown'], 'timeout': 25})")
else:
    client = Olostep(api_key=OLOSTEP_API_KEY)

    search = client.searches.create(
        query="What are the most important recent developments in AI agents for business research?",
        limit=5,
        scrape_options={"formats": ["markdown"], "timeout": 25},
    )

    for link in search.links:
        print(link["url"], "-", len(link.get("markdown_content") or ""), "chars")


https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai - 59839 chars
https://hbr.org/2025/11/the-ai-tools-that-are-transforming-market-research - 230 chars
https://mitsloan.mit.edu/ideas-made-to-matter/4-new-studies-about-agentic-ai-mit-initiative-digital-economy - 21844 chars
https://www.bcg.com/capabilities/artificial-intelligence/ai-agents - 28639 chars
https://www.mindstudio.ai/blog/ai-agents-research-analysis/ - 23150 chars


## Helpers

These helpers keep SDK responses compact and JSON-friendly for agent tool outputs.

Each Olostep helper also creates an OpenAI Agents trace span, so the trace shows which retrieval endpoint ran.


In [4]:
class OlostepError(RuntimeError):
    """Raised when an Olostep SDK request fails."""


def require_olostep_key() -> str:
    if not OLOSTEP_API_KEY:
        raise OlostepError("OLOSTEP_API_KEY is not set. Add it to .env and rerun the setup cell.")
    return OLOSTEP_API_KEY


def get_olostep_client() -> Olostep:
    return Olostep(api_key=require_olostep_key())


def sdk_result_to_dict(result: Any) -> dict[str, Any]:
    if hasattr(result, "model_dump"):
        return result.model_dump()
    if hasattr(result, "__dict__"):
        return {key: value for key, value in vars(result).items() if not key.startswith("_")}
    return {"value": str(result)}


def compact_json(data: Any, max_chars: int = 8000) -> str:
    text = json.dumps(data, ensure_ascii=False, indent=2, default=str)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n... [truncated]"


def normalize_search_links(links: list[dict[str, Any]], limit: int = 8) -> list[dict[str, Any]]:
    rows = []
    for link in links[:limit]:
        markdown = link.get("markdown_content") or ""
        rows.append({
            "title": link.get("title") or "Untitled",
            "url": link.get("url") or "",
            "description": link.get("description") or "",
            "markdown_chars": len(markdown),
            "markdown_preview": markdown[:1500] if markdown else "",
        })
    return rows


## Pydantic Structured Outputs

These models make the judge, source research, and final research report predictable.


In [5]:
class Judgment(BaseModel):
    is_good_enough: bool = Field(description="Whether the answer is sufficient for the user query.")
    score: float = Field(ge=0, le=1, description="Quality score from 0 to 1.")
    reason: str = Field(description="Short explanation of the decision.")
    missing_information: list[str] = Field(default_factory=list, description="Important gaps to fix.")


class SourceResearchReport(BaseModel):
    key_findings: list[str] = Field(description="Concise findings from gathered sources.")
    important_urls: list[str] = Field(description="Only the most important URLs used for synthesis.")
    source_notes: list[str] = Field(description="Brief notes connecting sources to findings.")
    remaining_gaps: list[str] = Field(default_factory=list, description="Gaps that could not be resolved.")


class MarkdownResearchReport(BaseModel):
    title: str = Field(description="Research report title.")
    executive_summary: str = Field(description="Short answer-first summary.")
    key_findings: list[str] = Field(description="Most important findings.")
    markdown_report: str = Field(description="Complete Markdown report with polished headings, clear analysis, reader-friendly structure, and citations.")
    citations: list[str] = Field(default_factory=list, description="Source URLs used in the report.")
    confidence: str = Field(description="Low, medium, or high confidence.")
    method_used: str = Field(description="Retrieval path used by the manager agent.")


## Olostep Tool Functions

These are the four requested functions. Each one is exposed as an OpenAI Agents SDK `function_tool`.


In [6]:
def _answer_query_impl(query: str) -> str:
    try:
        with custom_span("olostep.answer_query", {"query": query}):
            result = get_olostep_client().answers.create(task=query)
            return compact_json(sdk_result_to_dict(result))
    except Exception as exc:
        raise OlostepError(f"Olostep Answer API failed: {exc}") from exc


def _search_web_impl(query: str, limit: int = 8) -> str:
    try:
        with custom_span("olostep.search_web", {"query": query, "limit": limit}):
            search = get_olostep_client().searches.create(query=query, limit=limit)
            data = sdk_result_to_dict(search)
            return compact_json({
                "query": query,
                "results": normalize_search_links(data.get("links", []), limit=limit),
                "raw": data,
            })
    except Exception as exc:
        raise OlostepError(f"Olostep Search API failed: {exc}") from exc


def _search_with_scrape_impl(query: str, limit: int = 5) -> str:
    try:
        with custom_span(
            "olostep.search_with_scrape",
            {"query": query, "limit": limit, "scrape_options": {"formats": ["markdown"], "timeout": 25}},
        ):
            search = get_olostep_client().searches.create(
                query=query,
                limit=limit,
                scrape_options={"formats": ["markdown"], "timeout": 25},
            )
            data = sdk_result_to_dict(search)
            return compact_json({
                "query": query,
                "results": normalize_search_links(data.get("links", []), limit=limit),
                "raw": data,
            }, max_chars=12000)
    except Exception as exc:
        raise OlostepError(f"Olostep Search with Scrape failed: {exc}") from exc


def _scrape_url_impl(url: str) -> str:
    try:
        with custom_span("olostep.scrape_url", {"url": url, "formats": ["markdown"]}):
            scrape = get_olostep_client().scrapes.create(url=url, formats=["markdown"])
            return compact_json({"url": url, "scrape": sdk_result_to_dict(scrape)}, max_chars=10000)
    except Exception as exc:
        raise OlostepError(f"Olostep Scrape API failed: {exc}") from exc


@function_tool
async def answer_query(query: str) -> str:
    """Answer a natural-language research query using Olostep Answer API."""
    return _answer_query_impl(query)


@function_tool
async def search_web(query: str, limit: int = 8) -> str:
    """Search the web using Olostep Search and return normalized results."""
    return _search_web_impl(query, limit=limit)


@function_tool
async def search_with_scrape(query: str, limit: int = 5) -> str:
    """Search the web and scrape each returned link using Olostep Search with Scrape."""
    return _search_with_scrape_impl(query, limit=limit)


@function_tool
async def scrape_url(url: str) -> str:
    """Scrape one URL with Olostep and return compact page content."""
    return _scrape_url_impl(url)


## Specialist Agents

The manager will call these agents as tools.

This is the orchestration pattern: specialized agents remain separate, but the manager controls when to invoke them.


In [7]:
MODEL = "gpt-5.4-mini"

judge_agent = Agent(
    name="Judge agent",
    model=MODEL,
    instructions=(
        "You judge whether the provided answer is good enough for the original research question. "
        "Reward direct, specific, source-backed answers. Reject vague, stale, or unsupported answers. "
        "Return only the structured judgment."
    ),
    output_type=Judgment,
)

source_research_agent = Agent(
    name="Source research agent",
    model=MODEL,
    instructions=(
        "You gather evidence for a research report using only the provided Olostep tools. "
        "First try search_with_scrape for the original query. If the scraped search result is weak, "
        "run two or three targeted search_web calls, select only the most important URLs, scrape those URLs, "
        "and summarize the evidence. Prefer official, primary, reputable, and recent sources. "
        "Return only the structured source research report."
    ),
    tools=[search_web, search_with_scrape, scrape_url],
    output_type=SourceResearchReport,
)

analyst_agent = Agent(
    name="Analyst agent",
    model=MODEL,
    instructions=(
        "You write a proper Markdown research report from the evidence. "
        "Write for a professional reader who wants a clear, polished research brief on any topic. "
        "Adapt the report to the user's question. The markdown_report must be substantial, easy to scan, and use these general sections only: "
        "Executive Summary, Key Findings, Context, Evidence Review, Detailed Analysis, Implications, Source Notes, and References. "
        "If the topic is event-driven, include timeline details inside Context or Detailed Analysis instead of adding a separate Timeline section. "
        "If the topic is comparative, include a compact comparison table inside Detailed Analysis. "
        "Do not include sections titled Limitations, Next Steps, Recommendations, or Action Items. "
        "Avoid bare caveats like 'I relied on...'. Instead, integrate source quality naturally in Source Notes. "
        "Use short paragraphs, bullets where helpful, and citations as Markdown links or URL bullets. "
        "Add enough context that a non-expert reader understands the issue, why it matters, and what evidence supports it. "
        "Return only the structured report."
    ),
    output_type=MarkdownResearchReport,
)


## Manager Agent As Orchestrator

Here the manager is given the specialist agents as tools. The notebook will run only this manager agent for the real research workflow.


In [8]:
judge_tool = judge_agent.as_tool(
    tool_name="judge_answer_quality",
    tool_description="Judge whether an answer or evidence is good enough for the original research question.",
)

source_research_tool = source_research_agent.as_tool(
    tool_name="run_source_research",
    tool_description="Run Olostep search-with-scrape, targeted searches, URL selection, and URL scraping to gather source evidence.",
)

analyst_tool = analyst_agent.as_tool(
    tool_name="write_markdown_research_report",
    tool_description="Write the final structured Markdown research report from the gathered evidence.",
)

manager_agent = Agent(
    name="Manager research agent",
    model=MODEL,
    instructions=(
        "You are the orchestrator for a multi-agent research assistant. Follow this exact policy:\n"
        "1. Always call answer_query first for the user's question.\n"
        "2. Call judge_answer_quality on that Answer API result.\n"
        "3. If the judge says the answer is good enough, call write_markdown_research_report using the answer result.\n"
        "4. If the judge says the answer is not good enough, call run_source_research. The source researcher must use search_with_scrape first, then targeted searches and scrape_url if needed.\n"
        "5. Call write_markdown_research_report using all evidence.\n"
        "6. Return a complete MarkdownResearchReport. Do not return a casual chat answer."
    ),
    tools=[answer_query, judge_tool, source_research_tool, analyst_tool],
    output_type=MarkdownResearchReport,
)


## Run The Orchestrated Research Assistant With Tracing

This is now a single top-level manager run. The manager decides which tools and specialist agents to call.

The run is wrapped in `trace(...)`, and each Olostep helper creates a `custom_span(...)`. After the run, the cell prints a trace ID and logs URL so you can inspect agent calls, tool calls, and Olostep spans in the OpenAI tracer.


In [9]:
def openai_trace_url(trace_id: str) -> str:
    return f"https://platform.openai.com/logs/trace?trace_id={trace_id}"


async def run_research_assistant(query: str) -> MarkdownResearchReport:
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is not set. Add it to .env and rerun the setup cell.")
    require_olostep_key()

    trace_id = gen_trace_id()
    print("OpenAI trace ID:", trace_id)
    print("OpenAI trace URL:", openai_trace_url(trace_id))

    prompt = f"""
Research question:
{query}

Return a polished, reader-friendly Markdown research report with substantial detail for the user's specific question. Follow the required workflow exactly:
- Answer API first.
- Judge agent second.
- If weak, source research agent with search_with_scrape, targeted search_web calls, URL selection, and scrape_url.
- Analyst agent writes the final Markdown report. Do not include Limitations or Next Steps sections.
"""
    with trace(
        workflow_name="multi_agent_research_assistant_olostep",
        trace_id=trace_id,
        metadata={"query": query, "notebook": "multi_agent_research_assistant_openai_agents_olostep"},
    ):
        with custom_span("manager.run", {"query": query}):
            result = await Runner.run(manager_agent, prompt, max_turns=20)

    flush_traces()
    print("Trace flushed. Open the URL above to inspect manager, specialist agents, tools, and Olostep spans.")
    return result.final_output


## Example Query

This live example is guarded. Set `RUN_LIVE_EXAMPLE=true` in `.env` to run it.


In [10]:
example_query = "What's going on with OpenAI's Sora shutting down?"

if missing:
    print("Skipping live example because these environment variables are missing:", ", ".join(missing))
elif not RUN_LIVE_EXAMPLE:
    print("Skipping live example. Set RUN_LIVE_EXAMPLE=true in .env to run it.")
else:
    report = await run_research_assistant(example_query)
    display(Markdown(report.markdown_report))


OpenAI trace ID: trace_0b43796d9286432a857ed4dc84e0f64f
OpenAI trace URL: https://platform.openai.com/logs/trace?trace_id=trace_0b43796d9286432a857ed4dc84e0f64f
Trace flushed. Open the URL above to inspect manager, specialist agents, tools, and Olostep spans.


# What’s Going on with OpenAI’s Sora Shutting Down?

## Executive Summary

OpenAI is discontinuing Sora as a standalone product and is moving users through a formal wind-down process. The company’s Help Center confirms the shutdown and explains what users should expect.[^1] Reporting from BBC, the *Los Angeles Times*, and TechCrunch frames the move as a product-level discontinuation driven by strategic and operational factors, not a random outage.[^2][^3][^4]

## What happened

OpenAI confirmed that Sora is being shut down as a consumer product.[^1] In other words, this is a real discontinuation, not just a temporary service interruption.

## Timeline

- OpenAI published its official discontinuation notice.[^1]
- News outlets then reported on the shutdown and its context.[^2][^3][^4]
- Users are being given a managed transition rather than an abrupt cutoff.[^1]

## Why it’s shutting down

The clearest official explanation is that OpenAI is refocusing on other priorities.[^2] Independent reporting adds that Sora likely faced business and operational pressures: it is expensive to run, difficult to scale, and part of a broader product strategy shift.[^4]

The *Los Angeles Times* also reported that the expected Disney-related business angle did not materialize as hoped, which weakened one possible foundation for the product.[^3]

## What users should expect

OpenAI’s notice indicates users should prepare for changing access and possible export or account-handling steps during the wind-down.[^1] The practical takeaway: if you have content in Sora, act before the discontinuation completes.

## Bottom line

Sora is being shut down as an active OpenAI product, and the move appears to be a deliberate business/strategy decision rather than a technical emergency.[^1][^2][^3][^4]

## Citations

[^1]: OpenAI Help Center, “What to know about the Sora discontinuation” — https://help.openai.com/en/articles/20001152-what-to-know-about-the-sora-discontinuation
[^2]: BBC News, “OpenAI to shut down Sora” — https://www.bbc.com/news/articles/c3w3e467ewqo
[^3]: Los Angeles Times, “OpenAI will shut down Sora: why, what to know” — https://www.latimes.com/entertainment-arts/business/story/2026-03-24/openai-will-shut-down-sora-why-what-to-know
[^4]: TechCrunch, “Why OpenAI really shut down Sora” — https://techcrunch.com/2026/03/29/why-openai-really-shut-down-sora/

![image.png](image.png)

## Save A Markdown Report

After running the live example, you can save the report to a `.md` file.


In [11]:
# After running the live example above:
output_path = "research_report.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(report.markdown_report)
print(f"Saved {output_path}")


Saved research_report.md


## Exercise

Try another query by changing `my_query` below, then set `RUN_LIVE_EXAMPLE=true` when you are ready to call the APIs.


In [12]:
# Exercise scaffold
# my_query = "What changed in the latest OpenAI Agents SDK release?"
# report = await run_research_assistant(my_query)
# display(Markdown(report.markdown_report))


## Common Pitfalls

- If `client.searches` is missing, upgrade the SDK with `pip install -U olostep` and restart the notebook kernel.
- If you see `olostep ... current: 1.0.4`, the active kernel is still using the old SDK; restart the kernel after running the install cell.
- Keep `RUN_LIVE_EXAMPLE=false` when validating structure only.
- The Agents SDK supports orchestration through patterns such as agents-as-tools and handoffs. This notebook uses agents-as-tools because the manager should keep control and produce one final report.
- Use the printed OpenAI trace URL to inspect each agent/tool step; notebook stdout will only show the final report and trace metadata.
- Ask for Markdown explicitly in the analyst instructions and structured output model, otherwise agent output may drift back to a short answer.
- Keep the report general-purpose and reader-focused: Executive Summary, Key Findings, Context, Evidence Review, Detailed Analysis, Implications, Source Notes, and References.
